# Move the column

Drives the actuated column through the SDK, and shows the three ways a motion
can be refused or cut short. Mirrors `move_column.py`.

**This moves the column, including a full-travel homing run. Make sure people
are clear of it before running any cell below.**

Each step is its own cell so you can stop between motions.

In [ ]:
import time

from r2_labs import client, rpc_api

# The backend box, which is where the RPC frontend runs -- not the box the
# column is physically wired to. On a split pair those are different machines.
host = "localhost"

robot = client.Robot(
    f"tcp://{host}:{rpc_api.DEFAULT_PORT}",
    query_server_address=f"tcp://{host}:{rpc_api.DEFAULT_QUERY_PORT}",
    training_server_address=f"tcp://{host}:{rpc_api.DEFAULT_MODEL_TRAINER_PORT}",
)

target_mm = 200.0
stop_target_mm = 500.0

In [ ]:
def report_state(label):
  state = robot.column.get_state()
  print(
      f"[{label}] height={state.height_mm:.1f}mm"
      f" direction={state.direction.name.lower()}"
      f" calibrated={state.calibrated} limits_enabled={state.limits_enabled}"
      f" connected={state.connected}"
  )
  return state


start = report_state("start")
assert start.connected, (
    "the backend has no link to the column: check the column service is up on"
    " the hardware box and enable_column is true in the selected config"
)

## 1. A move before calibration is refused

Without a position reference the firmware has no idea where the column is, so
a move is rejected outright rather than accepted and quietly ignored. The
`error` comes back on the initiating call, and the ticket is already failed.

This only demonstrates anything on an uncalibrated column -- a position
reference cannot be discarded on demand, so power-cycle the board to see it.

In [ ]:
if robot.column.get_state().calibrated:
  print("already calibrated; power-cycle the board to see this case")
else:
  response = robot.column.initiate_go_to(target_mm)
  print(f"error: {response.error}")
  print(f"ticket: {robot.column.wait_for_ticket(response.ticket_id).info}")

## 2. Homing, and a move refused while it runs

Homing owns the motor for the length of the run, so a move issued during it is
refused for being busy. This starts homing, proves the refusal, then waits for
homing to finish -- about a minute of full-travel motion.

In [ ]:
calibration = robot.column.initiate_calibrate()
assert not calibration.error, calibration.error
print(f"homing started, ticket {calibration.ticket_id}")

# The firmware reports the motor busy only once homing is actually under way.
time.sleep(1.0)
refused = robot.column.initiate_go_to(target_mm)
print(f"move refused mid-homing: {refused.error}")

In [ ]:
robot.column.wait_for_ticket(calibration.ticket_id, timeout=90.0)
report_state("homed")

## 3. A move that completes

The ordinary case. `go_to` returns a future that resolves once the column has
stopped at the target.

In [ ]:
robot.column.go_to(target_mm, timeout=90.0).result()
report_state("arrived")

## 4. Stopping a move part way

`stop` cuts motor drive immediately. The move never reached its target, so its
ticket fails and waiting on it raises -- deliberate stops have to expect that.

In [ ]:
response = robot.column.initiate_go_to(stop_target_mm)
assert not response.error, response.error

time.sleep(2.0)
robot.column.stop()
print("stop() issued")

try:
  robot.column.wait_for_ticket(response.ticket_id, timeout=90.0)
except client.BehaviourFailedError as exc:
  print(f"ticket failed, as expected after a stop: {exc}")

report_state("stopped")

## Clearing a fault

A stall, an overcurrent or a thermal trip latches a lockout that refuses
further motion until it is cleared. `force=True` also clears a thermal one,
which is worth understanding before using: it tells a controller that believed
itself too hot to try again.

In [ ]:
print(robot.column.clear_fault(force=False))
report_state("fault cleared")